In [16]:
import pandas as pd
movies=pd.read_csv('tmdb_5000_movies.csv')
movies=movies[['id','title','overview','genres']]
movies.dropna(inplace=True)
movies.head()

,id,title,overview,genres
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."


In [21]:
import ast
def convert(obj):
  L=[]
  for i in ast.literal_eval(obj):
    L.append(i['name'])
  return L
movies['genres']=movies['genres'].apply(convert)
movies.head()

,id,title,overview,genres
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]"
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]"
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]"
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]"
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]"


In [27]:
movies['overview']=movies['overview'].apply(lambda x: x.split() if isinstance(x, str) else x)
movies['tags']=movies['genres']+movies['overview']
movies['tags']=movies['tags'].apply(lambda x:' '.join(x))
movies[['title','tags']].head()

,title,tags
0,Avatar,Action Adventure Fantasy Science Fiction In th...
1,Pirates of the Caribbean: At World's End,"Adventure Fantasy Action Captain Barbossa, lon..."
2,Spectre,Action Adventure Crime A cryptic message from ...
3,The Dark Knight Rises,Action Crime Drama Thriller Following the deat...
4,John Carter,Action Adventure Science Fiction John Carter i...


In [28]:
from sklearn.feature_extraction.text import CountVectorizer
#looks at top 5000 most common words and ignore stop words(like the is and etc)
cv=CountVectorizer(max_features=5000,stop_words='english')
vectors=cv.fit_transform(movies['tags']).toarray()
print(vectors.shape)

(643, 5000)


In [29]:
from sklearn.metrics.pairwise import cosine_similarity
#calulates similarity b/w every movie and every other movie
similarity=cosine_similarity(vectors)
print(similarity[0])

[1.         0.1860521  0.0860663  0.05976143 0.14704292 0.1490712
 0.02911113 0.23596995 0.09325048 0.1118034  0.20080483 0.10476454
 0.16269784 0.06523281 0.21516574 0.06523281 0.16064387 0.13187609
 0.0904534  0.09583148 0.07644708 0.08304548 0.10540926 0.14301939
 0.08944272 0.         0.2076137  0.18786729 0.18257419 0.09302605
 0.11338934 0.2795085  0.06388766 0.21081851 0.         0.21693046
 0.20701967 0.09128709 0.08385255 0.15118579 0.0766965  0.13187609
 0.         0.153393   0.07905694 0.14940358 0.23836565 0.15214515
 0.06030227 0.         0.12247449 0.1        0.17213259 0.06819943
 0.06454972 0.07547319 0.2        0.         0.06454972 0.10397505
 0.         0.25       0.13483997 0.06900656 0.1118034  0.03333333
 0.03535534 0.14509525 0.2795085  0.0451754  0.153393   0.11028219
 0.2236068  0.06454972 0.186339   0.07559289 0.15811388 0.
 0.09325048 0.14142136 0.05163978 0.06776309 0.14301939 0.20916501
 0.13975425 0.09759001 0.07352146 0.16903085 0.09759001 0.04016097
 0.1

In [33]:
def recommend(movie):
  # Find the index of the movie
  movie_index = movies[movies['title'] == movie].index[0]

  # Get similarity scores for that movie index
  distances = similarity[movie_index]

  # Sort by highest similarity and get top 5 (excluding the movie itself)
  movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

  for i in movies_list:
    print(movies.iloc[i[0]].title)
    # The AI only knows row numbers. Take the number [i[0]] and look up the actual title so a huamn can read it

recommend('Iron Man')

Iron Man 3
Avengers: Age of Ultron
Iron Man 2
Star Wars: Episode I - The Phantom Menace
Avatar
